In [2]:
!pip install transformers torchaudio soundfile


In [3]:
import torch
import torchaudio
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import soundfile as sf

c:\Users\zayna\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Load model and processor
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-960h")

c:\Users\zayna\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\zayna\.cache\huggingface\hub\models--facebook--wav2vec2-large-960h. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at face

In [12]:
# Load and preprocess audio (mono, 16kHz)
def load_audio(file_path):
    audio_input, sample_rate = sf.read(file_path)
    if sample_rate != 16000:
        raise ValueError("Audio must be sampled at 16kHz")
    
    # Convert stereo to mono if needed
    if len(audio_input.shape) == 2:
        audio_input = audio_input.mean(axis=1)
    
    return audio_input


# Transcribe audio
def transcribe(audio_path):
    input_audio = load_audio(audio_path)
    inputs = processor(input_audio, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]
    return transcription.lower()

In [13]:
import torchaudio

# Load the audio
waveform, sample_rate = torchaudio.load("/Users/zayna/Downloads/enhanced_audio.wav")

# Resample if needed
resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
resampled_waveform = resampler(waveform)

# Save the resampled audio
torchaudio.save("/Users/zayna/Downloads/enhanced_audio_16k.wav", resampled_waveform, 16000)

In [15]:
audio_path = "/Users/zayna/Downloads/enhanced_audio_16k.wav"
text = transcribe(audio_path)
print(text)


a catto on the er i a am a a a  a my an the go rit in the kitchen comfable he is he told to man he well no i new  en a i's the point am gone awie wel


In [16]:
!pip install datasets

   ---------------------------------------- 0.0/25.3 MB ? eta -:--:--
   - -------------------------------------- 0.8/25.3 MB 5.6 MB/s eta 0:00:05
   -- ------------------------------------- 1.8/25.3 MB 4.4 MB/s eta 0:00:06
   --- ------------------------------------ 2.4/25.3 MB 4.2 MB/s eta 0:00:06
   ----- ---------------------------------- 3.4/25.3 MB 4.0 MB/s eta 0:00:06
   ------- -------------------------------- 4.5/25.3 MB 4.3 MB/s eta 0:00:05
   --------- ------------------------------ 6.0/25.3 MB 4.8 MB/s eta 0:00:05
   ----------- ---------------------------- 7.1/25.3 MB 5.1 MB/s eta 0:00:04
   ------------- -------------------------- 8.7/25.3 MB 5.2 MB/s eta 0:00:04
   --------------- ------------------------ 9.7/25.3 MB 5.2 MB/s eta 0:00:03
   ---------------- ----------------------- 10.5/25.3 MB 5.1 MB/s eta 0:00:03
   ------------------ --------------------- 11.8/25.3 MB 5.2 MB/s eta 0:00:03
   ------------------- -------------------- 12.6/25.3 MB 5.2 MB/s eta 0:00:03
   

In [18]:
!pip install evaluate

In [20]:
!pip install jiwer

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.6 MB 1.0 MB/s eta 0:00:02
   ------------------- -------------------- 0.8/1.6 MB 1.2 MB/s eta 0:00:01
   ------------------------- -------------- 1.0/1.6 MB 1.4 MB/s eta 0:00:01
   ------------------------- -------------- 1.0/1.6 MB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 1.2 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.7
    Uninstalling click-8.1.7:
      Successfully uninstalled click-8.1.7


In [22]:
import evaluate

# Load WER and CER metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

# Your ground truth transcription (from image)
reference = """
mum
hello I'm here
okay
hello
hi
where's the cigarettes
in the kitchen
the camera's on
yes
are you talking to it while you work
no 
what y' doing then
what's the point oh god look what I'm wearing
"""

# Model prediction (your output from earlier)
prediction = """
a catto on the er i a am a a a  a my an the go rit in the kitchen comfable he is he told to man he well no i new  en a i's the point am gone awie wel
"""

# Normalize and flatten text
reference = " ".join(reference.lower().split())
prediction = " ".join(prediction.lower().split())

# Compute and display metrics
wer = wer_metric.compute(predictions=[prediction], references=[reference])
cer = cer_metric.compute(predictions=[prediction], references=[reference])

print(f"WER (Word Error Rate): {wer:.2%}")
print(f"CER (Character Error Rate): {cer:.2%}")


WER (Word Error Rate): 97.44%
CER (Character Error Rate): 58.46%
